# Entrega da Programação para a STS

Gera um arquivo `.txt` com as atividades solicitadas, sobe ao SharePoint e registra
metadados em `lake_relatorios_gerados.dbo.prog_enviada`.

**Fonte:** `lake_gold_fatos.dbo.base`  
**Entrada:** `atividade_ids_str` (CSV, injetado pelo Power Automate)  
**Saída:** arquivo `.txt` no SharePoint + linha na tabela `prog_enviada`

In [ ]:
# ── Parâmetros (Power Automate injeta valores em produção) ─────────────────
atividade_ids_str = '63000016479883,63000018896693,63000018461299'
solicitante       = ''   # email de quem solicitou (injetado pelo PA)
relatorio_id      = ''   # UUID gerado pelo PA antes do POST; gerado internamente se vazio
tipo_relatorio    = 'Entrega da Programação para a STS'
LAKEHOUSE_FOLDER  = 'Files/entrega_progs'

# Lakehouse onde os arquivos são gravados (lake_relatorios_gerados)
ONELAKE_WORKSPACE = 'ab5f231b-0d65-4a82-9fa3-e490336e912c'
ONELAKE_LAKEHOUSE = '59d32952-1672-4252-8c46-ff074684ed8e'

In [ ]:
import json
import struct
import unicodedata
import uuid
import tempfile
import warnings
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd

warnings.filterwarnings('ignore')

BRT = timedelta(hours=-3)

# ── Detecção de ambiente ───────────────────────────────────────────────────
try:
    spark
    FABRIC_ENV = True
    print('Ambiente: Microsoft Fabric')
except NameError:
    FABRIC_ENV = False
    print('Ambiente: local')

try:
    from notebookutils import mssparkutils as _ms
    mssparkutils = _ms
    HAS_MSSPARKUTILS = True
except ImportError:
    HAS_MSSPARKUTILS = False

if not FABRIC_ENV:
    try:
        import pyodbc
        from azure.identity import InteractiveBrowserCredential, DeviceCodeCredential
        _HAS_PYODBC = True
    except ImportError:
        _HAS_PYODBC = False
        print('[AVISO] pyodbc / azure-identity nao disponiveis.')
else:
    _HAS_PYODBC = False

SQL_ENDPOINT = (
    'beu5bmmdbuwedpv62ucm524jzi-dmrv7k3fbwbevh5d4sidg3urfq'
    '.datawarehouse.fabric.microsoft.com'
)

print('Imports OK -', (datetime.utcnow() + BRT).strftime('%d/%m/%Y %H:%M'))

In [ ]:
# ── Helpers de autenticacao ───────────────────────────────────────────────

def _get_db_conn():
    try:
        cred = InteractiveBrowserCredential()
    except Exception:
        cred = DeviceCodeCredential()
    token = cred.get_token('https://database.windows.net/.default').token
    tb = token.encode('utf-16-le')
    ts = struct.pack(f'<I{len(tb)}s', len(tb), tb)
    return pyodbc.connect(
        f'DRIVER={{ODBC Driver 17 for SQL Server}};'
        f'SERVER={SQL_ENDPOINT};Encrypt=Yes;',
        attrs_before={1256: ts},
    )


print('Helpers de autenticacao definidos.')

In [ ]:
# ── Busca campos de base + dim_unidade (todos os campos do relatório) ──────

def fetch_atividades(ids: list) -> pd.DataFrame:
    ids_sql = ', '.join(str(int(i)) for i in ids)
    if FABRIC_ENV:
        sql = f"""
            SELECT
                b.atividade_id,
                b.nome,
                b.custo_total,
                b.custo_contratos_total,
                b.gerencia,
                b.PrimeiraData        AS dataPrimeiraSessao,
                b.areaprog            AS area,
                b.linguagem,
                b.mes,
                b.autonomia,
                b.complemento,
                b.item_desc,
                b.projeto_nome        AS projeto,
                b.precificacao_desc,
                b.justificativa,
                du.unidade
            FROM lake_gold_fatos.dbo.base b
            LEFT JOIN lake_gold_fatos.dbo.dim_unidade du
                   ON CAST(LEFT(CAST(CAST(b.atividade_id AS BIGINT) AS STRING), 2) AS INT) = du.uo
            WHERE b.atividade_id IN ({ids_sql})
        """
        df = spark.sql(sql).toPandas()
    else:
        conn = _get_db_conn()
        sql_local = f"""
            SELECT
                b.atividade_id, b.nome, b.custo_total, b.custo_contratos_total,
                b.gerencia, b.PrimeiraData AS dataPrimeiraSessao,
                b.areaprog AS area, b.linguagem, b.mes, b.autonomia,
                b.complemento, b.item_desc, b.projeto_nome AS projeto,
                b.precificacao_desc, b.justificativa,
                (SELECT MIN(du.unidade)
                   FROM lake_gold_fatos.dbo.dim_unidade du
                  WHERE CAST(LEFT(CAST(CAST(b.atividade_id AS BIGINT) AS VARCHAR(20)), 2) AS INT) = du.uo) AS unidade
            FROM lake_gold_fatos.dbo.base b
            WHERE b.atividade_id IN ({ids_sql})
        """
        df = pd.read_sql(sql_local, conn)
        conn.close()

    df['atividade_id']           = pd.to_numeric(df['atividade_id'],           errors='coerce').astype('Int64')
    df['custo_total']            = pd.to_numeric(df['custo_total'],            errors='coerce').fillna(0.0)
    df['custo_contratos_total'] = pd.to_numeric(df['custo_contratos_total'], errors='coerce').fillna(0.0)
    return df


print('fetch_atividades definida.')

In [ ]:
# ── Busca contagem de sessões e local principal por atividade ─────────────

def fetch_sessoes(ids: list) -> pd.DataFrame:
    ids_sql = ', '.join(str(int(i)) for i in ids)
    if FABRIC_ENV:
        sql = f"""
            WITH local_cnt AS (
                SELECT atividade_id, localNome, COUNT(*) AS cnt
                FROM lake_gold_fatos.dbo.datas_sessoes
                WHERE atividade_id IN ({ids_sql})
                GROUP BY atividade_id, localNome
            ),
            local_ranked AS (
                SELECT atividade_id, localNome, cnt,
                       ROW_NUMBER() OVER (PARTITION BY atividade_id ORDER BY cnt DESC) AS rn
                FROM local_cnt
            ),
            sess_total AS (
                SELECT atividade_id, SUM(cnt) AS qt_sessoes
                FROM local_cnt
                GROUP BY atividade_id
            )
            SELECT st.atividade_id, st.qt_sessoes, lr.localNome AS localNome_max
            FROM sess_total st
            LEFT JOIN local_ranked lr
                   ON st.atividade_id = lr.atividade_id AND lr.rn = 1
        """
        df = spark.sql(sql).toPandas()
    else:
        df = pd.DataFrame(columns=['atividade_id', 'qt_sessoes', 'localNome_max'])

    df['atividade_id']  = pd.to_numeric(df['atividade_id'], errors='coerce').astype('Int64')
    df['qt_sessoes']    = pd.to_numeric(df['qt_sessoes'],   errors='coerce').fillna(0).astype(int)
    df['localNome_max'] = df['localNome_max'].fillna('')
    return df


print('fetch_sessoes definida.')

In [ ]:
# ── Busca dados de solicitações agrupados por atividade, grupo e complemento

GRUPOS_CONTRATO   = ('Contrato PJ', 'Contrato PF', 'Contrato Cooperativa')
GRUPOS_PASSAGEM   = ('Passagem Aérea',)
GRUPOS_HOSPEDAGEM = ('Hospedagem',)

def fetch_solicitacoes(ids: list) -> pd.DataFrame:
    ids_sql   = ', '.join(str(int(i)) for i in ids)
    g_cont    = ', '.join(f"'{g}'" for g in GRUPOS_CONTRATO)
    g_pass    = ', '.join(f"'{g}'" for g in GRUPOS_PASSAGEM)
    g_hosp    = ', '.join(f"'{g}'" for g in GRUPOS_HOSPEDAGEM)

    if FABRIC_ENV:
        sql = f"""
            SELECT
                atividade_id,
                grupo,
                complemento,
                SUM(custo) AS custo_grupo,
                MAX(CASE WHEN alerta = 1 THEN 1 ELSE 0 END) AS tem_alerta,
                COUNT(*) AS n_solic,
                SUM(CASE
                        WHEN grupo IN ({g_cont})
                         AND (complemento IS NULL OR LENGTH(TRIM(COALESCE(complemento, ''))) < 3)
                        THEN 1 ELSE 0
                    END) AS sem_pcap,
                SUM(CASE
                        WHEN grupo IN ({g_pass}, {g_hosp})
                         AND (publico_sessao IS NULL OR publico_sessao = 0)
                        THEN 1 ELSE 0
                    END) AS sem_pax
            FROM lake_gold_fatos.dbo.solicitacoes
            WHERE atividade_id IN ({ids_sql})
            GROUP BY atividade_id, grupo, complemento
        """
        df = spark.sql(sql).toPandas()
    else:
        df = pd.DataFrame(columns=[
            'atividade_id', 'grupo', 'complemento', 'custo_grupo',
            'tem_alerta', 'n_solic', 'sem_pcap', 'sem_pax'
        ])

    df['atividade_id'] = pd.to_numeric(df['atividade_id'], errors='coerce').astype('Int64')
    for col in ('custo_grupo', 'tem_alerta', 'n_solic', 'sem_pcap', 'sem_pax'):
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    df['complemento'] = df['complemento'].fillna('')
    return df


print('fetch_solicitacoes definida.')

In [ ]:
# ── Limiares para Projetos Relevantes (ajuste para testes) ────────────────
PROJ_MIN_ACOES = 10          # produção: 10
PROJ_MIN_TOTAL = 100_000     # produção: 100_000

# Mensagens de alerta por grupo de PAX (usadas em ambas as funções de relatório)
_MSG_PAX = {
    'Passagem Aérea': 'falta informar passageiros e/ou trechos',
    'Hospedagem':     'falta informar trechos',
}

# ── Helpers de formatação ─────────────────────────────────────────────────

_MES_NAMES = {
    1: 'Janeiro', 2: 'Fevereiro', 3: 'Março',    4: 'Abril',
    5: 'Maio',    6: 'Junho',     7: 'Julho',     8: 'Agosto',
    9: 'Setembro',10: 'Outubro',  11: 'Novembro', 12: 'Dezembro',
}

_MES_ABREV = {
    'jan': 1, 'fev': 2, 'mar': 3, 'abr': 4,
    'mai': 5, 'jun': 6, 'jul': 7, 'ago': 8,
    'set': 9, 'out': 10, 'nov': 11, 'dez': 12,
}


def _mes_to_int(v):
    """int, str numérica ou abreviação PT → inteiro (0 se desconhecido)."""
    try:
        return int(v)
    except (ValueError, TypeError):
        return _MES_ABREV.get(str(v).lower().strip(), 0)


def _brl(v):
    """Formata float como R$ x.xxx,xx (padrão PT-BR)."""
    return f'R$ {float(v):,.2f}'.replace(',', 'X').replace('.', ',').replace('X', '.')


def _fmt_data(v):
    """Converte qualquer valor de data para dd/mm/yyyy ou '' se inválido."""
    try:
        return pd.to_datetime(v, errors='coerce').strftime('%d/%m/%Y')
    except Exception:
        return str(v or '')


def _tabela_txt(headers, rows):
    """Tabela alinhada à esquerda para saída texto."""
    all_rows = [list(headers)] + [list(r) for r in rows]
    widths = [max(len(str(row[i])) for row in all_rows) for i in range(len(headers))]
    sep = '-+-'.join('-' * w for w in widths)
    lines = [' | '.join(str(headers[i]).ljust(widths[i]) for i in range(len(headers))), sep]
    for row in rows:
        lines.append(' | '.join(str(row[i]).ljust(widths[i]) for i in range(len(headers))))
    return '\n'.join(lines)


def _ativ_agr_txt(df_m, col):
    """Agrega df (com qt_sessoes) por coluna; retorna DataFrame ordenado por total desc."""
    return (
        df_m.groupby(col, dropna=False)
        .agg(n_acoes=('atividade_id', 'count'),
             qt_sessoes=('qt_sessoes', 'sum'),
             contratos=('custo_contratos_total', 'sum'),
             total=('custo_total', 'sum'))
        .reset_index()
        .sort_values('total', ascending=False)
    )


def _build_ativ_alertas(df_m, df_solic):
    """Constrói dict {atividade_id: [lista de alertas específicos]}."""
    ativ_alertas = {}

    if not df_solic.empty:
        for aid_g, grp in df_solic.groupby('atividade_id'):
            msgs = []
            for _, sr in grp.iterrows():
                if sr['sem_pcap'] > 0:
                    msgs.append(f'{sr["grupo"]}: falta informar PCAP')
                if sr['sem_pax'] > 0:
                    msg_pax = _MSG_PAX.get(sr['grupo'], 'falta informar passageiros e/ou trechos')
                    msgs.append(f'{sr["grupo"]}: {msg_pax}')
                if sr['tem_alerta'] > 0 and sr['sem_pcap'] == 0 and sr['sem_pax'] == 0:
                    msgs.append(f'{sr["grupo"]}: alerta na solicitação')
            if msgs:
                # deduplica mantendo ordem
                seen = set()
                ativ_alertas[int(aid_g)] = [m for m in msgs if not (m in seen or seen.add(m))]

    for _, r in df_m.iterrows():
        aid = int(r['atividade_id'])
        just = str(r.get('justificativa') or '').strip()
        if not just or just.upper() in ('NAN', 'NONE'):
            ativ_alertas.setdefault(aid, []).append('justificativa não informada')
        prec = str(r.get('precificacao_desc') or '').strip().upper()
        if not prec or prec in ('S/I', 'NAN'):
            ativ_alertas.setdefault(aid, []).append('precificação vazia ou S/I')

    return ativ_alertas


# ── Geração do relatório .txt ─────────────────────────────────────────────

def gerar_relatorio_txt(df_ativ, df_sess, df_solic, path, solicitante='', now_brt=None):
    if now_brt is None:
        now_brt = datetime.utcnow() + BRT

    df_m = df_ativ.copy()
    if 'qt_sessoes' not in df_m.columns:
        df_m = df_m.merge(df_sess[['atividade_id', 'qt_sessoes', 'localNome_max']], on='atividade_id', how='left')
        df_m['qt_sessoes']    = df_m['qt_sessoes'].fillna(0).astype(int)
        df_m['localNome_max'] = df_m['localNome_max'].fillna('')

    # Unidade para o cabeçalho
    u_vals = df_m['unidade'].dropna()
    unidade_titulo = str(u_vals.iloc[0]).strip() if len(u_vals) > 0 else ''

    out = []

    # Cabeçalho
    out.append('=' * 60)
    if unidade_titulo:
        out.append(f'Sesc {unidade_titulo}')
    out.append('REVISÃO PARA ENTREGA DA PROGRAMAÇÃO PARA A STS')
    out.append(f'Gerado em {now_brt.strftime("%d/%m/%Y %H:%M")} por {solicitante or "(não informado)"}')
    out.append('=' * 60)
    out.append('')

    # Resumo
    n_acoes = len(df_m)
    n_ger   = df_m['gerencia'].nunique()
    tot_c   = df_m['custo_contratos_total'].sum()
    tot_g   = df_m['custo_total'].sum()
    meses_c = df_m['mes'].dropna().map(_mes_to_int).value_counts().sort_index()
    meses_s = ', '.join(f'{_MES_NAMES.get(m, str(m))}: {c}' for m, c in meses_c.items())

    out.append('RESUMO')
    out.append('-' * 40)
    out.append(
        f'O filtro realizado contém {n_acoes} ações, para {n_ger} gerências, somando\n'
        f'{_brl(tot_c)} em contratos e {_brl(tot_g)} considerando todos os custos lançados.'
    )
    if meses_s:
        out.append(f'Meses: {meses_s}')
    out.append('')

    # Por gerência
    out.append('POR GERÊNCIA')
    out.append('-' * 40)
    grp = _ativ_agr_txt(df_m, 'gerencia')
    rows = [[r['gerencia'], r['n_acoes'], int(r['qt_sessoes']), _brl(r['contratos']), _brl(r['total'])]
            for _, r in grp.iterrows()]
    out.append(_tabela_txt(['Gerência', 'Ações', 'Sessões', 'Contratos', 'Total'], rows))
    out.append('')

    # Por linguagem
    out.append('POR LINGUAGEM')
    out.append('-' * 40)
    grp = _ativ_agr_txt(df_m, 'linguagem')
    rows = [[r['linguagem'], r['n_acoes'], int(r['qt_sessoes']), _brl(r['contratos']), _brl(r['total'])]
            for _, r in grp.iterrows()]
    out.append(_tabela_txt(['Linguagem', 'Ações', 'Sessões', 'Contratos', 'Total'], rows))
    out.append('')

    # Por item de custo
    out.append('POR ITEM DE CUSTO')
    out.append('-' * 40)
    if not df_solic.empty:
        sc = df_solic.merge(df_m[['atividade_id', 'qt_sessoes']].drop_duplicates(), on='atividade_id', how='left')
        gs = (sc.groupby('grupo', dropna=False)
                .agg(n_acoes=('atividade_id', 'nunique'),
                     qt_sessoes=('qt_sessoes', 'sum'),
                     valor=('custo_grupo', 'sum'))
                .reset_index()
                .sort_values('valor', ascending=False))
        rows = [[r['grupo'], r['n_acoes'], int(r['qt_sessoes']), _brl(r['valor'])]
                for _, r in gs.iterrows()]
        out.append(_tabela_txt(['Grupo', 'Ações', 'Sessões', 'Valor'], rows))
    else:
        out.append('(sem dados de solicitações)')
    out.append('')

    # Projetos relevantes — sem NaN, com limiares configuráveis
    out.append(f'PROJETOS RELEVANTES (>{PROJ_MIN_ACOES} ações ou >R$ {PROJ_MIN_TOTAL:,.0f})')
    out.append('-' * 40)
    df_proj = df_m[df_m['projeto'].notna() & (df_m['projeto'].astype(str).str.strip() != '')]
    proj = (df_proj
               .groupby('projeto')
               .agg(n_acoes=('atividade_id', 'count'),
                    qt_sessoes=('qt_sessoes', 'sum'),
                    contratos=('custo_contratos_total', 'sum'),
                    total=('custo_total', 'sum'))
               .reset_index())
    dest = proj[(proj['n_acoes'] > PROJ_MIN_ACOES) | (proj['total'] > PROJ_MIN_TOTAL)].sort_values('total', ascending=False)
    if not dest.empty:
        rows = [[r['projeto'], r['n_acoes'], int(r['qt_sessoes']), _brl(r['contratos']), _brl(r['total'])]
                for _, r in dest.iterrows()]
        out.append(_tabela_txt(['Projeto', 'Ações', 'Sessões', 'Contratos', 'Total'], rows))
    else:
        out.append('(nenhum projeto acima dos limites)')
    out.append('')

    # Alertas (totalizadores)
    alertas = []
    if not df_solic.empty:
        for grupo, cnt in df_solic[df_solic['sem_pcap'] > 0].groupby('grupo')['sem_pcap'].sum().items():
            alertas.append(f'[!] {int(cnt)} solicitação(ões) de "{grupo}" — falta informar PCAP')
        for grupo, cnt in df_solic[df_solic['sem_pax'] > 0].groupby('grupo')['sem_pax'].sum().items():
            msg = _MSG_PAX.get(grupo, 'falta informar passageiros e/ou trechos')
            alertas.append(f'[!] {int(cnt)} solicitação(ões) de "{grupo}" — {msg}')

    n_just = int((df_m['justificativa'].isna() | (df_m['justificativa'].astype(str).str.strip() == '')).sum())
    if n_just > 0:
        alertas.append(f'[!] {n_just} atividade(s) com o campo justificativa vazio')

    prec_up = df_m['precificacao_desc'].astype(str).str.strip().str.upper()
    n_prec  = int((df_m['precificacao_desc'].isna() | prec_up.isin(['', 'S/I', 'NAN'])).sum())
    if n_prec > 0:
        alertas.append(f'[!] {n_prec} atividade(s) com precificação vazia ou S/I')

    out.append('=' * 60)
    out.append('ALERTAS')
    out.append('=' * 60)
    out.extend(alertas if alertas else ['Nenhum alerta identificado.'])
    out.append('')

    # Dict de alertas por atividade
    ativ_alertas = _build_ativ_alertas(df_m, df_solic)

    # ── Bloco de ações por autonomia ──────────────────────────────────────
    def _bloco(df_sub, titulo, autonomia_val):
        blk = []
        is_uo = str(autonomia_val).strip().upper() == 'UO'

        blk.append('')
        blk.append(f'{titulo} ' + '=' * max(3, 58 - len(str(titulo))))
        blk.append('')

        for ger, df_ger in df_sub.groupby('gerencia', sort=True):
            ger_str = str(ger)
            blk.append(ger_str + ' ' + '-' * max(3, 50 - len(ger_str)))
            blk.append('')

            df_ger_s = df_ger.copy()
            df_ger_s['_dt_sort'] = pd.to_datetime(df_ger_s['dataPrimeiraSessao'], errors='coerce')
            df_ger_s = df_ger_s.sort_values('_dt_sort')

            for _, r in df_ger_s.iterrows():
                aid_int    = int(r['atividade_id'])
                aid_alerts = ativ_alertas.get(aid_int, [])
                flag       = ' [!]' if aid_alerts else ''
                dt_str     = _fmt_data(r.get('dataPrimeiraSessao'))
                qt         = int(r['qt_sessoes'])
                sess_str   = 'sessão única' if qt == 1 else f'{qt} sessões'

                if is_uo:
                    if aid_alerts:
                        alertas_inline = ' | '.join(aid_alerts)
                        blk.append(f'{dt_str}  {sess_str}  {r["nome"]}  {_brl(r["custo_total"])} [!] {alertas_inline}')
                    else:
                        blk.append(f'{dt_str}  {sess_str}  {r["nome"]}  {_brl(r["custo_total"])}')
                else:
                    nome = str(r.get('nome') or '')
                    comp = str(r.get('complemento') or '').strip()
                    nome_comp = (f'{nome} — {comp}'
                                 if comp and comp.upper() not in ('NAN', 'NONE')
                                 else nome)

                    area = str(r.get('area') or '').strip()
                    ling = str(r.get('linguagem') or '').strip()
                    area_ling = (area if area.lower() == ling.lower()
                                 else ' | '.join(filter(None, [area, ling])))

                    proj = str(r.get('projeto') or '').strip()

                    local = str(r.get('localNome_max') or '').strip()
                    prec  = str(r.get('precificacao_desc') or '').strip()
                    if prec.upper() in ('NAN', 'NONE', 'S/I'):
                        prec = ''
                    local_prec = ' — '.join(filter(None, [local, prec]))

                    blk.append(f'{dt_str}{flag}')
                    blk.append(sess_str)
                    blk.append(nome_comp)
                    if proj and proj.upper() not in ('NAN', 'NONE'):
                        blk.append(proj)
                    if area_ling:
                        blk.append(area_ling)

                    # Itens de custo do df_solic — grupo + complemento + valor
                    if not df_solic.empty:
                        ativ_solic = df_solic[df_solic['atividade_id'] == r['atividade_id']]
                        for _, sr in ativ_solic.sort_values('custo_grupo', ascending=False).iterrows():
                            comp_s = str(sr.get('complemento') or '').strip()
                            if comp_s and comp_s.upper() not in ('NAN', 'NONE'):
                                blk.append(f'{sr["grupo"]}  {comp_s}  {_brl(sr["custo_grupo"])}')
                            else:
                                blk.append(f'{sr["grupo"]}  {_brl(sr["custo_grupo"])}')

                    if local_prec:
                        blk.append(local_prec)
                    for alrt in aid_alerts:
                        blk.append(f'[!] {alrt}')
                    blk.append('')

        return blk

    for aut in sorted(df_m['autonomia'].dropna().unique()):
        out.extend(_bloco(df_m[df_m['autonomia'] == aut], aut, aut))
    sem_aut = df_m[df_m['autonomia'].isna()]
    if not sem_aut.empty:
        out.extend(_bloco(sem_aut, 'SEM AUTONOMIA', 'SEM AUTONOMIA'))

    path.write_text('\n'.join(out), encoding='utf-8')


print('_brl, _tabela_txt, gerar_relatorio_txt definidos.')

In [ ]:
# ── Geração do relatório .html (para corpo do e-mail no Power Automate) ───

_TH  = 'background:#1f4e79;color:white;padding:6px 10px;text-align:left;border:1px solid #ccc;'
_TD  = 'padding:5px 10px;border:1px solid #ccc;'
_TDR = 'padding:5px 10px;border:1px solid #ccc;text-align:right;'
_TBL = 'border-collapse:collapse;width:100%;margin-bottom:16px;font-family:Arial,sans-serif;font-size:13px;'
_H2  = 'font-family:Arial,sans-serif;color:#1f4e79;margin-top:24px;margin-bottom:4px;'
_H3  = 'font-family:Arial,sans-serif;color:#1f4e79;margin-top:20px;margin-bottom:4px;'
_P   = 'font-family:Arial,sans-serif;font-size:13px;margin:4px 0;'


def _html_table(headers, rows, alert_idxs=None):
    alert_idxs = alert_idxs or set()
    h = f'<table style="{_TBL}"><thead><tr>'
    for hdr in headers:
        h += f'<th style="{_TH}">{hdr}</th>'
    h += '</tr></thead><tbody>'
    for i, row in enumerate(rows):
        bg = 'background:#fff2cc;' if i in alert_idxs else ''
        h += f'<tr style="{bg}">'
        for cell in row:
            s = str(cell)
            td = _TDR if (s.startswith('R$') or s.replace('.', '').replace(',', '').isdigit()) else _TD
            h += f'<td style="{td}{bg}">{s}</td>'
        h += '</tr>'
    h += '</tbody></table>'
    return h


def gerar_relatorio_html(df_ativ, df_sess, df_solic, solicitante='', now_brt=None):
    if now_brt is None:
        now_brt = datetime.utcnow() + BRT

    df_m = df_ativ.copy()
    if 'qt_sessoes' not in df_m.columns:
        df_m = df_m.merge(df_sess[['atividade_id', 'qt_sessoes', 'localNome_max']], on='atividade_id', how='left')
        df_m['qt_sessoes']    = df_m['qt_sessoes'].fillna(0).astype(int)
        df_m['localNome_max'] = df_m['localNome_max'].fillna('')

    # Unidade para o cabeçalho
    u_vals = df_m['unidade'].dropna()
    unidade_titulo = str(u_vals.iloc[0]).strip() if len(u_vals) > 0 else ''
    titulo_prefix = f'Sesc {unidade_titulo} — ' if unidade_titulo else ''

    p = []

    # Cabeçalho
    p.append(f'<h2 style="{_H2}">{titulo_prefix}Revisão para Entrega da Programação para a STS</h2>')
    p.append(
        f'<p style="{_P}">Gerado em <strong>{now_brt.strftime("%d/%m/%Y %H:%M")}</strong>'
        f' por <strong>{solicitante or "(não informado)"}</strong></p>'
    )

    # Resumo
    n_acoes = len(df_m)
    n_ger   = df_m['gerencia'].nunique()
    tot_c   = df_m['custo_contratos_total'].sum()
    tot_g   = df_m['custo_total'].sum()
    meses_c = df_m['mes'].dropna().map(_mes_to_int).value_counts().sort_index()
    meses_s = ', '.join(f'{_MES_NAMES.get(m, str(m))}: {c}' for m, c in meses_c.items())

    p.append(f'<h3 style="{_H3}">Resumo</h3>')
    p.append(
        f'<p style="{_P}">O filtro realizado contém <strong>{n_acoes}</strong> ações, '
        f'para <strong>{n_ger}</strong> gerências, somando '
        f'<strong>{_brl(tot_c)}</strong> em contratos e '
        f'<strong>{_brl(tot_g)}</strong> considerando todos os custos lançados.</p>'
    )
    if meses_s:
        p.append(f'<p style="{_P}"><strong>Meses:</strong> {meses_s}</p>')

    def _agr(col):
        return (df_m.groupby(col, dropna=False)
                    .agg(n_acoes=('atividade_id', 'count'),
                         qt_sessoes=('qt_sessoes', 'sum'),
                         contratos=('custo_contratos_total', 'sum'),
                         total=('custo_total', 'sum'))
                    .reset_index()
                    .sort_values('total', ascending=False))

    # Por gerência
    p.append(f'<h3 style="{_H3}">Por Gerência</h3>')
    g = _agr('gerencia')
    rows = [[r['gerencia'], r['n_acoes'], int(r['qt_sessoes']), _brl(r['contratos']), _brl(r['total'])]
            for _, r in g.iterrows()]
    p.append(_html_table(['Gerência', 'Ações', 'Sessões', 'Contratos', 'Total'], rows))

    # Por linguagem
    p.append(f'<h3 style="{_H3}">Por Linguagem</h3>')
    g = _agr('linguagem')
    rows = [[r['linguagem'], r['n_acoes'], int(r['qt_sessoes']), _brl(r['contratos']), _brl(r['total'])]
            for _, r in g.iterrows()]
    p.append(_html_table(['Linguagem', 'Ações', 'Sessões', 'Contratos', 'Total'], rows))

    # Por item de custo
    p.append(f'<h3 style="{_H3}">Por Item de Custo</h3>')
    if not df_solic.empty:
        sc = df_solic.merge(df_m[['atividade_id', 'qt_sessoes']].drop_duplicates(), on='atividade_id', how='left')
        gs = (sc.groupby('grupo', dropna=False)
                .agg(n_acoes=('atividade_id', 'nunique'),
                     qt_sessoes=('qt_sessoes', 'sum'),
                     valor=('custo_grupo', 'sum'))
                .reset_index()
                .sort_values('valor', ascending=False))
        rows = [[r['grupo'], r['n_acoes'], int(r['qt_sessoes']), _brl(r['valor'])]
                for _, r in gs.iterrows()]
        p.append(_html_table(['Grupo', 'Ações', 'Sessões', 'Valor'], rows))
    else:
        p.append(f'<p style="{_P};color:#888;">(sem dados de solicitações)</p>')

    # Projetos relevantes — sem NaN, com limiares configuráveis
    p.append(f'<h3 style="{_H3}">Projetos Relevantes (&gt;{PROJ_MIN_ACOES} ações ou &gt;R$&nbsp;{PROJ_MIN_TOTAL:,.0f})</h3>')
    df_proj = df_m[df_m['projeto'].notna() & (df_m['projeto'].astype(str).str.strip() != '')]
    proj = (df_proj
               .groupby('projeto')
               .agg(n_acoes=('atividade_id', 'count'),
                    qt_sessoes=('qt_sessoes', 'sum'),
                    contratos=('custo_contratos_total', 'sum'),
                    total=('custo_total', 'sum'))
               .reset_index())
    dest = proj[(proj['n_acoes'] > PROJ_MIN_ACOES) | (proj['total'] > PROJ_MIN_TOTAL)].sort_values('total', ascending=False)
    if not dest.empty:
        rows = [[r['projeto'], r['n_acoes'], int(r['qt_sessoes']), _brl(r['contratos']), _brl(r['total'])]
                for _, r in dest.iterrows()]
        p.append(_html_table(['Projeto', 'Ações', 'Sessões', 'Contratos', 'Total'], rows))
    else:
        p.append(f'<p style="{_P};color:#888;">(nenhum projeto acima dos limites)</p>')

    # Alertas (totalizadores)
    alertas_html = []
    if not df_solic.empty:
        for grupo, cnt in df_solic[df_solic['sem_pcap'] > 0].groupby('grupo')['sem_pcap'].sum().items():
            alertas_html.append(f'<strong>{int(cnt)}</strong> solicitação(ões) de "{grupo}" — falta informar PCAP')
        for grupo, cnt in df_solic[df_solic['sem_pax'] > 0].groupby('grupo')['sem_pax'].sum().items():
            msg = _MSG_PAX.get(grupo, 'falta informar passageiros e/ou trechos')
            alertas_html.append(f'<strong>{int(cnt)}</strong> solicitação(ões) de "{grupo}" — {msg}')

    n_just = int((df_m['justificativa'].isna() | (df_m['justificativa'].astype(str).str.strip() == '')).sum())
    if n_just > 0:
        alertas_html.append(f'<strong>{n_just}</strong> atividade(s) com o campo justificativa vazio')

    prec_up = df_m['precificacao_desc'].astype(str).str.strip().str.upper()
    n_prec  = int((df_m['precificacao_desc'].isna() | prec_up.isin(['', 'S/I', 'NAN'])).sum())
    if n_prec > 0:
        alertas_html.append(f'<strong>{n_prec}</strong> atividade(s) com precificação vazia ou S/I')

    if alertas_html:
        p.append('<h3 style="font-family:Arial,sans-serif;color:#c00000;margin-top:20px;">Alertas</h3>')
        p.append('<ul style="font-family:Arial,sans-serif;font-size:13px;color:#c00000;">')
        p.extend(f'<li>{a}</li>' for a in alertas_html)
        p.append('</ul>')

    # Dict de alertas por atividade
    ativ_alertas = _build_ativ_alertas(df_m, df_solic)

    HDR_ACAO = ['#', 'ID', 'Nome', 'Gerência', 'Área', 'Ling.', 'Mês', 'Unidade',
                '1ª Sessão', 'Sessões', 'Projeto', 'Item/Complemento', 'Local',
                'Justificativa', 'Contratos', 'Total', 'Alertas']

    def _bloco_html(df_sub, titulo):
        bl = [f'<h3 style="{_H3}">{titulo} ({len(df_sub)} ações)</h3>']
        if df_sub.empty:
            bl.append(f'<p style="{_P};color:#888;">(nenhuma ação)</p>')
            return ''.join(bl)
        # Ordena por gerência e depois por data crescente dentro de cada gerência
        _ds = df_sub.copy()
        _ds['_dt_sort'] = pd.to_datetime(_ds['dataPrimeiraSessao'], errors='coerce')
        df_sub = _ds.sort_values(['gerencia', '_dt_sort']).drop(columns=['_dt_sort'])
        rows, alert_idxs = [], set()
        for i, (_, r) in enumerate(df_sub.reset_index(drop=True).iterrows(), 1):
            aid = int(r['atividade_id'])
            local_prec = ' — '.join(filter(None, [
                str(r.get('localNome_max') or '').strip(),
                str(r.get('precificacao_desc') or '').strip(),
            ]))
            item_comp = ' / '.join(filter(None, [
                str(r.get('item_desc') or '').strip(),
                str(r.get('complemento') or '').strip(),
            ]))
            just_txt = str(r.get('justificativa') or '').strip()
            if just_txt.upper() in ('NAN', 'NONE'):
                just_txt = ''
            aid_alerts = ativ_alertas.get(aid, [])
            alertas_str = ' | '.join(aid_alerts)
            rows.append([
                i, aid, r['nome'], r['gerencia'], r['area'], r['linguagem'],
                r['mes'], r['unidade'], _fmt_data(r['dataPrimeiraSessao']), int(r['qt_sessoes']),
                r.get('projeto') or '', item_comp, local_prec, just_txt,
                _brl(r['custo_contratos_total']), _brl(r['custo_total']), alertas_str,
            ])
            if aid_alerts:
                alert_idxs.add(i - 1)
        bl.append(_html_table(HDR_ACAO, rows, alert_idxs))
        return ''.join(bl)

    for aut in sorted(df_m['autonomia'].dropna().unique()):
        p.append(_bloco_html(df_m[df_m['autonomia'] == aut], f'Ações — {aut}'))
    sem_aut = df_m[df_m['autonomia'].isna()]
    if not sem_aut.empty:
        p.append(_bloco_html(sem_aut, 'Ações — SEM AUTONOMIA'))

    return '\n'.join(p)


print('gerar_relatorio_html definida.')

In [ ]:
# ── Salva arquivo no Lakehouse (PA faz o upload para o SharePoint) ─────────

def salvar_lakehouse(local_path, folder):
    dest = folder.rstrip('/') + '/' + local_path.name
    if HAS_MSSPARKUTILS:
        abfss = (
            f'abfss://{ONELAKE_WORKSPACE}'
            f'@onelake.dfs.fabric.microsoft.com'
            f'/{ONELAKE_LAKEHOUSE}/{dest}'
        )
        content_bytes = local_path.read_bytes()
        jvm  = spark._jvm
        conf = spark._jsc.hadoopConfiguration()
        path = jvm.org.apache.hadoop.fs.Path(abfss)
        fs   = path.getFileSystem(conf)
        out  = fs.create(path, True)  # True = overwrite
        out.write(content_bytes)
        out.close()
        print(f'Arquivo gravado em {abfss}')
    else:
        import shutil
        out = Path(tempfile.gettempdir()) / 'lh_sim' / local_path.name
        out.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(local_path, out)
        print(f'[LOCAL] arquivo copiado para {out}')
    return dest


print('salvar_lakehouse definida.')

In [ ]:
# ── Registro em lake_relatorios_gerados.dbo.prog_enviada ─────────────────

def registrar_relatorio(relatorio_id, relatorio_nome, url,
                        ids, qt_total, gerencias, solicitante):
    row = pd.DataFrame([{
        'relatorio_id':   relatorio_id,
        'relatorio_nome': relatorio_nome,
        'data_geracao':   datetime.now(),
        'url_arquivo':    url,
        'solicitante':    solicitante,
        'atividade_ids':  ','.join(str(i) for i in ids),
        'qt_total':       qt_total,
        'gerencias':      gerencias,
    }])

    if FABRIC_ENV:
        (spark.createDataFrame(row)
              .write.mode('append')
              .option('mergeSchema', 'true')
              .saveAsTable('lake_relatorios_gerados.dbo.prog_enviada'))
        print('Linha registrada em lake_relatorios_gerados.dbo.prog_enviada.')
    else:
        print('LOCAL - registro que seria gravado em lake_relatorios_gerados.dbo.prog_enviada:')
        print(row.to_string(index=False))


print('registrar_relatorio definida.')

In [ ]:
# ── Execução principal ────────────────────────────────────────────────────

raw = atividade_ids_str.strip()
if raw.startswith('{'):
    raw = '[' + raw + ']'

if raw.startswith('['):
    parsed = json.loads(raw)
    if parsed and isinstance(parsed[0], dict):
        ids = [int(x['atividade_id']) for x in parsed]
    else:
        ids = [int(x) for x in parsed]
else:
    ids = [int(x.strip()) for x in raw.split(',') if x.strip()]

if not ids:
    raise ValueError(f'atividade_ids_str está vazio ou inválido: {repr(atividade_ids_str)}')

print(f'{len(ids)} atividades: {ids}')

# 1. Consultas
df = fetch_atividades(ids)
print(f'Dados base: {len(df)} linha(s)')
print(df[['atividade_id', 'nome', 'custo_total', 'gerencia', 'autonomia']].to_string(index=False))

df_sess  = fetch_sessoes(ids)
df_solic = fetch_solicitacoes(ids)
print(f'Sessões: {len(df_sess)} linha(s) | Solicitações: {len(df_solic)} linha(s)')

# 2. Merge sessões em df
df = df.merge(df_sess[['atividade_id', 'qt_sessoes', 'localNome_max']], on='atividade_id', how='left')
df['qt_sessoes']    = df['qt_sessoes'].fillna(0).astype(int)
df['localNome_max'] = df['localNome_max'].fillna('')

# 3. Unidade, gerências e timestamp BRT
unidade_raw   = df['unidade'].iloc[0] if len(df) > 0 and pd.notna(df['unidade'].iloc[0]) else 'SEM_UO'
unidade_ascii = unicodedata.normalize('NFKD', str(unidade_raw)).encode('ascii', 'ignore').decode('ascii')
unidade_safe  = ''.join(c if c.isalnum() else '_' for c in unidade_ascii).strip('_')
gerencias_str = '|'.join(sorted(df['gerencia'].dropna().unique().tolist()))
now_brt = datetime.utcnow() + BRT
ts_str  = now_brt.strftime('%Y%m%d_%H%M%S')
print(f'Unidade: {unidade_raw} | Gerências: {gerencias_str}')

# 4. UUID do relatório — usa o injetado pelo PA (garante isolamento por execução)
rid = relatorio_id.strip() if relatorio_id.strip() else str(uuid.uuid4())

# 5. Gera e salva .txt
filename_txt = f'relatorio_sts_{unidade_safe}_{ts_str}.txt'
tmp_txt = Path(tempfile.gettempdir()) / filename_txt
gerar_relatorio_txt(df, df_sess, df_solic, tmp_txt, solicitante=solicitante, now_brt=now_brt)
print(f'TXT gerado: {tmp_txt}')
lh_path_txt = salvar_lakehouse(tmp_txt, LAKEHOUSE_FOLDER)

# 6. Gera e salva .html
filename_html = f'relatorio_sts_{unidade_safe}_{ts_str}.html'
tmp_html = Path(tempfile.gettempdir()) / filename_html
tmp_html.write_text(
    gerar_relatorio_html(df, df_sess, df_solic, solicitante=solicitante, now_brt=now_brt),
    encoding='utf-8',
)
print(f'HTML gerado: {tmp_html}')
lh_path_html = salvar_lakehouse(tmp_html, LAKEHOUSE_FOLDER)

# 7. Monta resultado e grava _output_{rid}.json — nome único por execução (evita race condition)
resultado = {
    'relatorio_id':   rid,
    'relatorio_nome': tipo_relatorio,
    'filename':       filename_txt,
    'lh_path':        lh_path_txt,
    'filename_html':  filename_html,
    'lh_path_html':   lh_path_html,
    'unidade':        unidade_raw,
    'gerencias':      gerencias_str,
    'qt_total':       len(ids),
    'gerado_em':      now_brt.isoformat(),
}
resultado_str = json.dumps(resultado, ensure_ascii=False)

if HAS_MSSPARKUTILS:
    output_json_name = f'_output_{rid}.json'
    mssparkutils.fs.put(LAKEHOUSE_FOLDER + '/' + output_json_name, resultado_str, overwrite=True)
    print(f'{output_json_name} gravado em {LAKEHOUSE_FOLDER}/')

print(resultado_str)

if HAS_MSSPARKUTILS:
    mssparkutils.notebook.exit(resultado_str)